# Pix2Pix training on Foggy Cityscapes (medium + dense)

**Phase 2** of the project: train two Pix2Pix generators that learn to
translate clean street images into foggy ones (medium fog and dense fog).

Once both generators are trained, in Phase 3 we will apply them to VDD
(aerial drone images) to obtain VDD_foggy_medium and VDD_foggy_dense, then
test the segmentation U-Net's robustness on these.

## Contents

1. Verify GPU
2. Clone code from GitHub
3. Mount Google Drive
4. Copy Foggy Cityscapes from Drive to local SSD
5. Install dependencies
6. Sanity check (smoke test on a tiny subset)
7. Launch TensorBoard
8. **Training: medium fog** (~30-40 min)
9. Backup medium-fog run to Drive
10. **Training: dense fog** (~30-40 min)
11. Backup dense-fog run to Drive
12. Final summary and sample grids

**Before running**: Runtime -> Change runtime type -> T4 GPU.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

In [ ]:
import os

!rm -rf /content/fog-uav-robustness
!git clone https://github.com/FabrizioCozzolino/fog-uav-robustness.git /content/fog-uav-robustness
%cd /content/fog-uav-robustness
!ls

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/fog-uav-robustness'
!ls -la "{DRIVE_PROJECT}/data"

Expected: you should see `Foggy_Cityscapes/` (the folder you uploaded) plus
`VDD/` (already there from Phase 1).

## 4. Copy Foggy Cityscapes from Drive to local SSD

Same logic as for VDD: reading thousands of small files directly from Drive
is slow during training. We copy once at session start.

Foggy Cityscapes is small (~500 MB total), so this takes ~3-5 minutes.

In [ ]:
import os, time, shutil

DRIVE_FC = '/content/drive/MyDrive/fog-uav-robustness/data/Foggy_Cityscapes'
LOCAL_FC = '/content/data/Foggy_Cityscapes'

assert os.path.isdir(DRIVE_FC), f'Foggy_Cityscapes not found at {DRIVE_FC}'

# Auto-detect possible nesting (Foggy_Cityscapes/Foggy_Cityscapes/...)
if os.path.isdir(os.path.join(DRIVE_FC, 'Foggy_Cityscapes', 'No_Fog')):
    src = os.path.join(DRIVE_FC, 'Foggy_Cityscapes')
    print(f'Detected nested layout, copying from {src}')
elif os.path.isdir(os.path.join(DRIVE_FC, 'No_Fog')):
    src = DRIVE_FC
    print(f'Flat layout, copying from {src}')
else:
    raise RuntimeError(f'Could not find No_Fog/ inside {DRIVE_FC}.')

if os.path.isdir(LOCAL_FC):
    print(f'Removing existing {LOCAL_FC} ...')
    shutil.rmtree(LOCAL_FC)
os.makedirs('/content/data', exist_ok=True)

print('Copying ... (~3-5 minutes for ~500 MB)')
t0 = time.time()
!cp -r "{src}" "{LOCAL_FC}"
print(f'Copy took {time.time() - t0:.0f} s')

In [ ]:
# Sanity check
import os
for sub in ['No_Fog', 'Medium_Fog', 'Dense_Fog']:
    d = os.path.join(LOCAL_FC, sub)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{sub:12s}: {n:4d} files')

Expected: `No_Fog: 500`, `Medium_Fog: 500`, `Dense_Fog: 500`.

## 5. Install Python dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
!python scripts/check_env.py

## 6. Pix2Pix smoke test

Before launching the long training, run the model smoke test on the GPU.
Should take 5-15 seconds and confirm both networks build, forward, backward.

In [ ]:
!python scripts/test_pix2pix.py

## 7. Launch TensorBoard

Open TB now so curves stream live as the training runs. Two runs will
appear (one per fog level).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/runs

## 8. Training: medium fog

Hyperparameters:

- `--epochs 50`: paper recommends 100-200 but 50 is enough for 450 paired images.
- `--batch-size 8`: comfortable on T4 at 256x256.
- `--lr 2e-4`, `--beta1 0.5`: Pix2Pix paper defaults.
- `--lambda-l1 100`: paper default; balances adversarial vs reconstruction.

Estimated time on T4: ~30-40 min.

In [ ]:
!python src/training/train_pix2pix.py \
    --data-root /content/data/Foggy_Cityscapes \
    --fog-level medium \
    --output-dir outputs/runs \
    --run-name pix2pix_medium_baseline \
    --encoder resnet34 \
    --image-size 256 \
    --epochs 50 \
    --batch-size 8 \
    --val-batch-size 8 \
    --num-workers 2 \
    --lr 2e-4 \
    --beta1 0.5 \
    --lambda-l1 100.0 \
    --sample-every 5 \
    --seed 42

## 9. Backup medium-fog run to Drive

Critical: when this Colab session ends, /content/ is wiped. Copy the run
(generator weights, samples, TB logs, history) to Drive so we don't lose it.

In [ ]:
import shutil, os

RUN_NAME = 'pix2pix_medium_baseline'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN_NAME}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN_NAME}'

if os.path.isdir(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print(f'[ok] backed up to Drive: {DST}')
!ls -la "{DST}"
!echo '---'
!ls -la "{DST}/samples" | head -20

## 10. Training: dense fog

Same configuration but `--fog-level dense`. The dense fog is harder for the
network because the transformation is more aggressive (more pixels change).
Estimated time on T4: ~30-40 min.

In [ ]:
!python src/training/train_pix2pix.py \
    --data-root /content/data/Foggy_Cityscapes \
    --fog-level dense \
    --output-dir outputs/runs \
    --run-name pix2pix_dense_baseline \
    --encoder resnet34 \
    --image-size 256 \
    --epochs 50 \
    --batch-size 8 \
    --val-batch-size 8 \
    --num-workers 2 \
    --lr 2e-4 \
    --beta1 0.5 \
    --lambda-l1 100.0 \
    --sample-every 5 \
    --seed 42

## 11. Backup dense-fog run to Drive

In [ ]:
import shutil, os

RUN_NAME = 'pix2pix_dense_baseline'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN_NAME}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN_NAME}'

if os.path.isdir(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print(f'[ok] backed up to Drive: {DST}')
!ls -la "{DST}"

## 12. Final summary and visualization

Print best val L1 for both runs and show the final epoch's sample grid.

In [ ]:
import json

for run_name in ['pix2pix_medium_baseline', 'pix2pix_dense_baseline']:
    history_path = f'/content/fog-uav-robustness/outputs/runs/{run_name}/history.json'
    if not os.path.isfile(history_path):
        print(f'[skip] {run_name}: no history.json')
        continue
    with open(history_path) as f:
        history = json.load(f)
    best = min(history, key=lambda e: e['val_L1'])
    last = history[-1]
    print('=' * 60)
    print(f'{run_name}')
    print('=' * 60)
    print(f'Total epochs            : {len(history)}')
    print(f'Total time              : {sum(e["time_s"] for e in history):.0f} s')
    print(f'Best epoch              : {best["epoch"]}')
    print(f'Best val L1             : {best["val_L1"]:.4f}')
    print(f'Final epoch loss_D      : {last["loss_D"]:.3f}')
    print(f'Final epoch G_l1        : {last["loss_G_l1"]:.3f}')
    print(f'Final epoch D(real)     : {last["D_real_mean"]:.2f}')
    print(f'Final epoch D(fake)     : {last["D_fake_mean"]:.2f}')
    print()

In [ ]:
# Display the final sample grids inline so we can see how the GANs did.
from IPython.display import Image, display
import os

for run_name in ['pix2pix_medium_baseline', 'pix2pix_dense_baseline']:
    samples_dir = f'/content/fog-uav-robustness/outputs/runs/{run_name}/samples'
    if not os.path.isdir(samples_dir):
        print(f'[skip] {run_name}: no samples dir')
        continue
    samples = sorted(os.listdir(samples_dir))
    if not samples:
        continue
    final_sample = os.path.join(samples_dir, samples[-1])
    print(f'\n=== {run_name} -- final sample: {samples[-1]} ===')
    print('Rows: clean (input) / fake (G output) / real (target)')
    display(Image(filename=final_sample))

**Reading the sample grids**:

- Row 1 (top): the **clean** input image. This is what we feed to G.
- Row 2 (middle): the **fake** foggy image generated by G.
- Row 3 (bottom): the **real** foggy image (the ground-truth target).

The closer Row 2 looks to Row 3, the better the GAN learned. With 50 epochs
on 450 paired images and L1 loss weighted 100x, expect the medium-fog G to
produce convincing results; the dense-fog G is harder because the
transformation is more aggressive.

## What's next (Phase 3)

After this notebook finishes:

1. Both `G_best.pth` files are on Drive (medium and dense).
2. We will write `inference/generate_foggy_vdd.py` that loads each generator
   and applies it to ALL VDD images, producing `VDD_foggy_medium/` and
   `VDD_foggy_dense/`. The segmentation labels stay identical.
3. We will run the U-Net (Phase 1's v2 model) on these foggy VDD sets and
   measure the mIoU drop -- the central result of the project.